# K3s Architecture (Dash Cytoscape)

This notebook cell recreates the `k3s-architecture.svg` layout with draggable nodes.

- Use a **Python 3.11** kernel.
- Run the code cell and open the provided local URL.
- Drag nodes to refine layout; stop the app with kernel interrupt when done.


In [4]:
# Python 3.11
from dash import Dash, html
import dash_cytoscape as cyto

APP_PORT = 8065

# Preset coordinates use the same visual balance as pictures/k3s-architecture.svg
nodes = [
    # ===== Compound frames =====
    {"data": {"id": "k3s_server", "label": "k3s Server"}, "classes": "frame-node", "selectable": False, "grabbable": False},
    {"data": {"id": "k3s_agent", "label": "k3s Agent"}, "classes": "frame-node", "selectable": False, "grabbable": False},

    {"data": {"id": "server_process", "label": "Process", "parent": "k3s_server"}, "classes": "process-node", "position": {"x": 430, "y": 160}, "selectable": False, "grabbable": False},
    {"data": {"id": "agent_process", "label": "Process", "parent": "k3s_agent"}, "classes": "process-node", "position": {"x": 1130, "y": 160}, "selectable": False, "grabbable": False},

    # ===== Server process nodes =====
    {"data": {"id": "supervisor", "label": "Supervisor", "parent": "server_process"}, "position": {"x": 260, "y": 220}, "classes": "k8s-node"},
    {"data": {"id": "api_server", "label": "API\nServer", "parent": "server_process"}, "position": {"x": 260, "y": 360}, "classes": "k8s-node"},
    {"data": {"id": "kine", "label": "Kine", "parent": "server_process"}, "position": {"x": 120, "y": 360}, "classes": "k8s-node"},
    {"data": {"id": "kube_proxy_server", "label": "Kube\nProxy", "parent": "server_process"}, "position": {"x": 420, "y": 360}, "classes": "k8s-node"},
    {"data": {"id": "flannel_server", "label": "Flannel", "parent": "server_process"}, "position": {"x": 580, "y": 360}, "classes": "k8s-node"},
    {"data": {"id": "scheduler", "label": "Scheduler", "parent": "server_process"}, "position": {"x": 120, "y": 520}, "classes": "k8s-node"},
    {"data": {"id": "controller_manager", "label": "Controller\nManager", "parent": "server_process"}, "position": {"x": 260, "y": 520}, "classes": "k8s-node"},
    {"data": {"id": "kubelet_server", "label": "Kubelet", "parent": "server_process"}, "position": {"x": 460, "y": 520}, "classes": "k8s-node"},

    # ===== Server runtime + pods =====
    {"data": {"id": "containerd_server", "label": "containerd", "parent": "k3s_server"}, "position": {"x": 460, "y": 650}, "classes": "runtime-node"},
    {"data": {"id": "pod_s_1", "label": "Pod", "parent": "k3s_server"}, "position": {"x": 320, "y": 790}, "classes": "pod-node"},
    {"data": {"id": "pod_s_2", "label": "Pod", "parent": "k3s_server"}, "position": {"x": 460, "y": 790}, "classes": "pod-node"},
    {"data": {"id": "pod_s_3", "label": "Pod", "parent": "k3s_server"}, "position": {"x": 600, "y": 790}, "classes": "pod-node"},

    # ===== Agent process nodes =====
    {"data": {"id": "tunnel_proxy", "label": "Tunnel\nProxy", "parent": "agent_process"}, "position": {"x": 980, "y": 360}, "classes": "k8s-node"},
    {"data": {"id": "kube_proxy_agent", "label": "Kube\nProxy", "parent": "agent_process"}, "position": {"x": 1140, "y": 360}, "classes": "k8s-node"},
    {"data": {"id": "flannel_agent", "label": "Flannel", "parent": "agent_process"}, "position": {"x": 1300, "y": 360}, "classes": "k8s-node"},
    {"data": {"id": "kubelet_agent", "label": "Kubelet", "parent": "agent_process"}, "position": {"x": 1140, "y": 520}, "classes": "k8s-node"},

    # ===== Agent runtime + pods =====
    {"data": {"id": "containerd_agent", "label": "containerd", "parent": "k3s_agent"}, "position": {"x": 1140, "y": 650}, "classes": "runtime-node"},
    {"data": {"id": "pod_a_1", "label": "Pod", "parent": "k3s_agent"}, "position": {"x": 1000, "y": 790}, "classes": "pod-node"},
    {"data": {"id": "pod_a_2", "label": "Pod", "parent": "k3s_agent"}, "position": {"x": 1140, "y": 790}, "classes": "pod-node"},
    {"data": {"id": "pod_a_3", "label": "Pod", "parent": "k3s_agent"}, "position": {"x": 1280, "y": 790}, "classes": "pod-node"},
]

edges = [
    # Server process
    {"data": {"id": "e_supervisor_api", "source": "supervisor", "target": "api_server"}},
    {"data": {"id": "e_api_kine", "source": "api_server", "target": "kine"}},
    {"data": {"id": "e_kproxy_api", "source": "kube_proxy_server", "target": "api_server"}},
    {"data": {"id": "e_flannel_kproxy_s", "source": "flannel_server", "target": "kube_proxy_server"}},
    {"data": {"id": "e_scheduler_api", "source": "scheduler", "target": "api_server"}},
    {"data": {"id": "e_cm_api", "source": "controller_manager", "target": "api_server"}},
    {"data": {"id": "e_kubelet_api", "source": "kubelet_server", "target": "api_server"}},
    {"data": {"id": "e_kubelet_flannel_s", "source": "kubelet_server", "target": "flannel_server"}},

    # Cross-server/agent tunnel line
    {"data": {"id": "e_supervisor_tunnel", "source": "supervisor", "target": "tunnel_proxy"}},

    # Agent process
    {"data": {"id": "e_flannel_kproxy_a", "source": "flannel_agent", "target": "kube_proxy_agent"}},
    {"data": {"id": "e_kproxy_tunnel", "source": "kube_proxy_agent", "target": "tunnel_proxy"}},
    {"data": {"id": "e_kubelet_tunnel", "source": "kubelet_agent", "target": "tunnel_proxy"}},
    {"data": {"id": "e_kubelet_flannel_a", "source": "kubelet_agent", "target": "flannel_agent"}},

    # Runtimes + pods
    {"data": {"id": "e_kubelet_containerd_s", "source": "kubelet_server", "target": "containerd_server"}},
    {"data": {"id": "e_containerd_s_p1", "source": "containerd_server", "target": "pod_s_1"}},
    {"data": {"id": "e_containerd_s_p2", "source": "containerd_server", "target": "pod_s_2"}},
    {"data": {"id": "e_containerd_s_p3", "source": "containerd_server", "target": "pod_s_3"}},

    {"data": {"id": "e_kubelet_containerd_a", "source": "kubelet_agent", "target": "containerd_agent"}},
    {"data": {"id": "e_containerd_a_p1", "source": "containerd_agent", "target": "pod_a_1"}},
    {"data": {"id": "e_containerd_a_p2", "source": "containerd_agent", "target": "pod_a_2"}},
    {"data": {"id": "e_containerd_a_p3", "source": "containerd_agent", "target": "pod_a_3"}},
]

elements = nodes + edges

stylesheet = [
    {
        "selector": "core",
        "style": {
            "background-color": "#FFF4D2"
        },
    },
    {
        "selector": "node",
        "style": {
            "label": "data(label)",
            "text-wrap": "wrap",
            "text-max-width": 120,
            "font-size": 16,
            "font-weight": 700,
            "font-family": "Arial, sans-serif",
            "text-valign": "center",
            "text-halign": "center",
            "background-color": "#F8E6A9",
            "border-color": "#33313B",
            "border-width": 0,
            "color": "#33313B",
            "shape": "round-rectangle",
            "width": 180,
            "height": 86,
        },
    },
    {
        "selector": ".runtime-node",
        "style": {
            "width": 180,
            "height": 86,
            "font-size": 38,
            "font-family": "Arial Black, Arial, sans-serif",
            "text-max-width": 170,
        },
    },
    {
        "selector": ".pod-node",
        "style": {
            "width": 95,
            "height": 95,
            "font-size": 40,
            "font-family": "Arial Black, Arial, sans-serif",
            "shape": "round-rectangle",
            "text-max-width": 80,
        },
    },
    {
        "selector": ".frame-node",
        "style": {
            "background-opacity": 0,
            "border-color": "#33313B",
            "border-width": 3,
            "shape": "round-rectangle",
            "padding": "38px",
            "text-valign": "top",
            "text-halign": "center",
            "font-size": 40,
            "font-family": "Arial Black, Arial, sans-serif",
            "font-weight": 800,
            "color": "#33313B",
            "z-index": 0,
        },
    },
    {
        "selector": ".process-node",
        "style": {
            "background-opacity": 0,
            "border-color": "#33313B",
            "border-width": 3,
            "border-style": "dashed",
            "shape": "round-rectangle",
            "padding": "26px",
            "text-valign": "top",
            "text-halign": "center",
            "font-size": 28,
            "font-family": "Arial Black, Arial, sans-serif",
            "font-weight": 800,
            "color": "#33313B",
            "z-index": 1,
        },
    },
    {
        "selector": "edge",
        "style": {
            "curve-style": "taxi",
            "taxi-direction": "vertical",
            "taxi-turn": 40,
            "line-color": "#33313B",
            "width": 3,
            "target-arrow-shape": "triangle",
            "target-arrow-color": "#33313B",
            "arrow-scale": 1.0,
        },
    },
]

app = Dash(__name__)
app.layout = html.Div(
    [
        cyto.Cytoscape(
            id="k3s-architecture-cytoscape",
            elements=elements,
            layout={"name": "preset", "padding": 20},
            stylesheet=stylesheet,
            style={"width": "100%", "height": "900px", "backgroundColor": "#FFF4D2"},
            minZoom=0.35,
            maxZoom=2.2,
            wheelSensitivity=0.2,
            autoungrabify=False,
            userPanningEnabled=True,
            userZoomingEnabled=True,
            boxSelectionEnabled=True,
        )
    ]
)

print(f"Starting Dash Cytoscape on http://127.0.0.1:{APP_PORT}")
print("Stop execution when you are done (Kernel Interrupt).")

try:
    # Dash 2.x (works in JupyterLab)
    app.run(debug=False, host="127.0.0.1", port=APP_PORT, jupyter_mode="external")
except TypeError:
    # Older Dash fallback
    app.run_server(debug=False, host="127.0.0.1", port=APP_PORT)


Starting Dash Cytoscape on http://127.0.0.1:8065
Stop execution when you are done (Kernel Interrupt).
Dash app running on http://127.0.0.1:8065/
